# **Malicious URL Detection**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import nltk
import re
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("sid321axn/malicious-urls-dataset")

path = os.path.join(path, os.listdir(path)[0])
print(path)

Using Colab cache for faster access to the 'malicious-urls-dataset' dataset.
/kaggle/input/malicious-urls-dataset/malicious_phish.csv


In [ ]:
df = pd.read_csv(path)
df

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement
...,...,...
651186,xbox360.ign.com/objects/850/850402.html,phishing
651187,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing
651188,www.gamespot.com/xbox360/action/deadspace/,phishing
651189,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing


In [ ]:
df['type'].value_counts()

,count
type,
benign,428103
defacement,96457
phishing,94111
malware,32520


In [ ]:
df_2_classified = df.replace({'type':{'benign':0, 'defacement':1, 'phishing':1, 'malware':1}}, inplace=False)

/tmp/ipython-input-584717917.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2_classified = df.replace({'type':{'benign':0, 'defacement':1, 'phishing':1, 'malware':1}}, inplace=False)


In [ ]:
df_2_classified['type'].value_counts()

,count
type,
0,428103
1,223088


In [ ]:
#Imbalancement Management

df_2_classified_benign = df_2_classified[df_2_classified['type'] == 0]
df_2_classified_malware = df_2_classified[df_2_classified['type'] == 1]

In [ ]:
print(df_2_classified_benign.shape)
print(df_2_classified_malware.shape)

(428103, 2)
(223088, 2)


In [ ]:
df_2_classified_benign = df_2_classified_benign.sample(n=223088, random_state=2)
print(df_2_classified_benign.shape)
print(df_2_classified_malware.shape)

(223088, 2)
(223088, 2)


In [ ]:
df_2_classified= pd.concat([df_2_classified_benign, df_2_classified_malware])
df_2_classified

,url,type
588202,goredirt.co.uk/identity.php?cmd=SignIn&co_part...,0
47247,theglobeandmail.com/life/travel/on-the-edge-at...,0
219869,cgs.illinois.edu/people/faculty/paul-diehl,0
310312,kenbarlow.net/,0
571258,ib2wbradesco.atendimentoseguro.ms11.net/2ident...,0
...,...,...
651186,xbox360.ign.com/objects/850/850402.html,1
651187,games.teamxbox.com/xbox-360/1860/Dead-Space/,1
651188,www.gamespot.com/xbox360/action/deadspace/,1
651189,en.wikipedia.org/wiki/Dead_Space_(video_game),1


In [ ]:
url_vec = TfidfVectorizer()

In [ ]:
X = df_2_classified['url']
Y = df_2_classified['type']
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, stratify=Y, random_state=2)

In [ ]:
X_train_vec = url_vec.fit_transform(X_train)
X_test_vec = url_vec.transform(X_test)


In [ ]:
model_lg = LogisticRegression()
model_lg.fit(X_train_vec, Y_train)

LogisticRegression()

In [ ]:
X_train_vec_pred = model_lg.predict(X_train_vec)
X_test_vec_pred = model_lg.predict(X_test_vec)
print('Training Accuracy : ', accuracy_score(X_train_vec_pred, Y_train))
print('Testing Accuracy : ', accuracy_score(X_test_vec_pred, Y_test))

Training Accuracy :  0.9509132420091324
Testing Accuracy :  0.9412339525209783
